# 🤖 Multi-Agent Systems with LangGraph
## Interview Kickstart — Agentic AI Series

**Topic:** Build a Multi-Agent System with LangGraph  
**Audience:** Software engineers (5+ years) transitioning to Generative AI  
**Time:** ~45 minutes to run through completely  

---

### What You'll Build
A 4-agent research pipeline that takes a user question and:
1. **Plans** — breaks it into focused subtasks
2. **Researches** — gathers information using tools
3. **Analyzes** — evaluates quality and synthesizes findings
4. **Writes** — produces a structured, cited report

### Prerequisites
- `OPENAI_API_KEY` set as environment variable
- Run `pip install -r ../code/requirements.txt` first

## Part 0: Setup & Configuration

In [ ]:
# Install required packages (run once)
# Uncomment if not already installed
# !pip install langchain langchain-openai langgraph langchain-community faiss-cpu python-dotenv -q

In [ ]:
import os
import sys
import json
from pathlib import Path

# Add code/ directory to path
CODE_DIR = Path("../code").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Load environment variables from .env file if present
from dotenv import load_dotenv
load_dotenv()
load_dotenv(CODE_DIR / ".env")

# Verify setup
assert os.getenv("OPENAI_API_KEY"), (
    "❌ OPENAI_API_KEY not set. "
    "Run: export OPENAI_API_KEY='sk-...' in your terminal, "
    "or create a .env file in the code/ directory."
)
print("✅ Setup complete")
print(f"   Python: {sys.version.split()[0]}")
print(f"   Working directory: {Path.cwd()}")

## Part 1: Core Concept — GraphState

Before we build agents, let's understand the **shared state** they communicate through.

### Why TypedDict?
- Type-safe: each field has a declared type
- Dict-like: agents update by returning `{"field": new_value}`
- Inspectable: you can print the entire state at any point
- JSON-serializable: enables checkpointing (saves state to disk)

In [ ]:
from typing import TypedDict, List, Optional

class GraphState(TypedDict):
    """
    The shared data structure that flows through every node in the LangGraph.
    Think of this as a shared whiteboard your team of agents can all read and write.
    """
    query: str                    # Original user question
    plan: List[str]               # Subtasks from Planner
    research_results: List[dict]  # Findings from Researcher
    additional_queries: List[str] # Analyst-requested follow-ups
    analysis: str                 # Synthesized insights from Analyst
    final_report: str             # Final output from Writer
    iteration_count: int          # Guard against infinite loops
    approved: bool                # Analyst's verdict on research quality

# Initial state — this is what we pass to the graph to kick things off
initial_state: GraphState = {
    "query": "What are the best practices for building production-ready LLM applications?",
    "plan": [],
    "research_results": [],
    "additional_queries": [],
    "analysis": "",
    "final_report": "",
    "iteration_count": 0,
    "approved": False,
}

print("Initial state keys:", list(initial_state.keys()))
print("\nQuery:", initial_state["query"])

## Part 2: RAG Pipeline — Building the Knowledge Base

Before agents start working, we load a knowledge base into a **vector store** (FAISS).
This allows the Researcher agent to do **semantic search** over local documents.

In [ ]:
from rag_pipeline import RAGPipeline
from tools import set_rag_pipeline

# Initialize RAG pipeline
rag = RAGPipeline(
    embedding_model="text-embedding-3-small",
    chunk_size=500,
    chunk_overlap=100,
    top_k=3
)

print(f"RAG pipeline: {rag}")
print(f"Ready: {rag.is_ready}")

In [ ]:
# Load the sample knowledge base
KB_PATH = Path("../demo/sample_data/knowledge_base.txt")

if KB_PATH.exists():
    n_chunks = rag.ingest(str(KB_PATH))
    print(f"\n✅ Indexed {n_chunks} chunks from: {KB_PATH.name}")
else:
    # Inline knowledge base as fallback
    text = """
    LangGraph is a framework for building stateful multi-agent AI systems as directed graphs.
    Key features: typed state, conditional edges, checkpointing, and streaming support.
    Multi-agent systems improve quality by specializing each agent to a focused task.
    RAG (Retrieval-Augmented Generation) reduces hallucination by grounding LLM answers in retrieved context.
    Best practices for LLM applications include output validation, retry logic, caching, and observability.
    """
    n_chunks = rag.ingest_text(text, source="inline")
    print(f"\n✅ Indexed {n_chunks} chunks from inline text")

# Test retrieval
print("\n🔍 Test retrieval for 'LangGraph features':")
print("-" * 40)
result = rag.retrieve("LangGraph key features")
print(result[:400] + "...")

In [ ]:
# Inject RAG pipeline into tools module (so agents can use it)
set_rag_pipeline(rag)
print("✅ RAG pipeline injected into tools module")

# Now test the rag_retrieve tool directly
from tools import rag_retrieve
result = rag_retrieve.invoke("production LLM best practices")
print("\n🔧 Tool test — rag_retrieve('production LLM best practices'):")
print(result[:300] + "...")

## Part 3: Individual Agent Nodes

Let's test each agent node in isolation before wiring them together.

### Why test nodes in isolation?
- Faster iteration: fix one node without running the full graph
- Easier debugging: see exactly what each agent produces
- Unit testing: each node is a pure function → easy to test

In [ ]:
from agent import planner_node, GraphState

# Test Planner in isolation
test_state = dict(initial_state)  # copy initial state

print("=" * 50)
print("PLANNER NODE TEST")
print("=" * 50)
print(f"Input query: {test_state['query']}\n")

planner_result = planner_node(test_state)

print(f"\n📋 Returned state update:")
for key, value in planner_result.items():
    print(f"  {key}: {value}")

In [ ]:
# Merge planner output into state (as LangGraph would do automatically)
test_state.update(planner_result)

print("State after Planner:")
print(f"  plan ({len(test_state['plan'])} items):")
for i, task in enumerate(test_state['plan'], 1):
    print(f"    {i}. {task}")

In [ ]:
from agent import researcher_node

# Test Researcher on just the FIRST subtask (for speed)
# In production, it processes all subtasks
research_test_state = dict(test_state)
research_test_state['plan'] = [test_state['plan'][0]]  # just first subtask

print("=" * 50)
print("RESEARCHER NODE TEST (1 subtask)")
print("=" * 50)

researcher_result = researcher_node(research_test_state)

print(f"\n📊 Research results ({len(researcher_result['research_results'])} items):")
for r in researcher_result['research_results']:
    print(f"\nSubtask: {r['subtask'][:80]}")
    print(f"Findings: {r['findings'][:200]}...")

In [ ]:
from agent import analyst_node

# Test Analyst with the research results we have
analyst_test_state = dict(test_state)
analyst_test_state.update(researcher_result)

print("=" * 50)
print("ANALYST NODE TEST")
print("=" * 50)

analyst_result = analyst_node(analyst_test_state)

print(f"\n📊 Analyst output:")
print(f"  approved: {analyst_result['approved']}")
print(f"  analysis: {analyst_result['analysis'][:300]}...")
if analyst_result.get('additional_queries'):
    print(f"  additional_queries: {analyst_result['additional_queries']}")

In [ ]:
from agent import writer_node

# Test Writer
writer_test_state = dict(analyst_test_state)
writer_test_state.update(analyst_result)
writer_test_state['approved'] = True  # Force approval for testing

print("=" * 50)
print("WRITER NODE TEST")
print("=" * 50)

writer_result = writer_node(writer_test_state)

print(f"\n📄 Report preview (first 800 chars):")
print(writer_result['final_report'][:800])
print("\n[...report continues...]")

## Part 4: Building the LangGraph Workflow

Now we wire the individual agent nodes together using LangGraph's `StateGraph`.

### Key concepts:
- `add_node`: register a Python function as a graph node
- `set_entry_point`: where execution begins
- `add_edge`: unconditional transition between nodes
- `add_conditional_edges`: branch based on a routing function's return value
- `compile()`: validate the graph and return an executable app

In [ ]:
from langgraph.graph import StateGraph, END
from agent import (
    GraphState,
    planner_node,
    researcher_node, 
    analyst_node,
    writer_node,
    should_continue,
)

# ---- Build the workflow graph ----------------------------------------
workflow = StateGraph(GraphState)

# Register each agent as a node
workflow.add_node("planner",    planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst",    analyst_node)
workflow.add_node("writer",     writer_node)

# Wire up the graph
workflow.set_entry_point("planner")           # Start here
workflow.add_edge("planner",    "researcher") # Always: planner → researcher
workflow.add_edge("researcher", "analyst")    # Always: researcher → analyst

# Conditional edge from analyst:
#   should_continue() returns "continue" or "done"
#   → "continue" maps to researcher (loop back)
#   → "done"     maps to writer (proceed)
workflow.add_conditional_edges(
    "analyst",
    should_continue,
    {
        "continue": "researcher",  # Loop: more research needed
        "done":     "writer",      # Proceed: write the report
    }
)

workflow.add_edge("writer", END)              # Writer is terminal

# Compile → validates graph structure and creates optimized executor
app = workflow.compile()

print("✅ LangGraph workflow compiled successfully")
print("\nGraph nodes:", list(workflow.nodes.keys()) if hasattr(workflow, 'nodes') else "[compiled]")

## Part 5: Running the Full Pipeline

Now let's run the complete multi-agent pipeline.

We'll use `app.stream()` instead of `app.invoke()` so we can see each node's output as it completes — this is important for:
- Demos: showing real-time progress
- Debugging: catching failures early
- Production: streaming partial results to the UI

In [ ]:
# Define the research query
QUERY = "What are the best practices for building production-ready LLM applications in 2025?"

initial_state = {
    "query": QUERY,
    "plan": [],
    "research_results": [],
    "additional_queries": [],
    "analysis": "",
    "final_report": "",
    "iteration_count": 0,
    "approved": False,
}

print("=" * 60)
print("🤖 MULTI-AGENT RESEARCH PIPELINE")
print("=" * 60)
print(f"Query: {QUERY}")
print("=" * 60)

# Store for later analysis
all_events = []
final_report = ""

# Stream execution — yields one event per node completion
for event in app.stream(initial_state):
    all_events.append(event)
    node_name = list(event.keys())[0]
    node_output = event[node_name]
    
    print(f"\n✅ Node completed: [{node_name.upper()}]")
    
    if "final_report" in node_output and node_output["final_report"]:
        final_report = node_output["final_report"]

print("\n" + "=" * 60)
print(f"Pipeline complete. {len(all_events)} nodes executed.")

In [ ]:
# Display the final report
from IPython.display import Markdown, display

print("=" * 60)
print("📄 FINAL RESEARCH REPORT")
print("=" * 60)

if final_report:
    display(Markdown(final_report))
else:
    print("No final report generated — check for errors above")

## Part 6: Inspecting Intermediate State

One of LangGraph's key advantages: you can inspect the state at every step.
This is critical for debugging and understanding what your agents are doing.

In [ ]:
print("Execution trace:\n")
for i, event in enumerate(all_events):
    node_name = list(event.keys())[0]
    node_output = event[node_name]
    
    print(f"Step {i+1}: [{node_name.upper()}]")
    print(f"  Keys updated: {list(node_output.keys())}")
    
    # Show relevant state updates
    if "plan" in node_output:
        print(f"  Plan: {len(node_output['plan'])} subtasks")
    if "research_results" in node_output:
        print(f"  Research: {len(node_output['research_results'])} results")
    if "approved" in node_output:
        print(f"  Approved: {node_output['approved']}")
    if "iteration_count" in node_output:
        print(f"  Iteration: {node_output['iteration_count']}")
    if "final_report" in node_output and node_output["final_report"]:
        wc = len(node_output["final_report"].split())
        print(f"  Report: {wc} words")
    print()

## Part 7: Understanding the Routing Function

The `should_continue()` function is the heart of the self-correcting loop.
Let's test different scenarios.

In [ ]:
from agent import should_continue

# Scenario 1: Analyst approves on first try
state_approved = {"iteration_count": 1, "approved": True}
print(f"Scenario 1 (approved, iter=1): should_continue → '{should_continue(state_approved)}'")
assert should_continue(state_approved) == "done"

# Scenario 2: Analyst rejects, needs more research
state_loop = {"iteration_count": 1, "approved": False}
print(f"Scenario 2 (rejected, iter=1): should_continue → '{should_continue(state_loop)}'")
assert should_continue(state_loop) == "continue"

# Scenario 3: Max iterations hit — force proceed to avoid infinite loop
state_max = {"iteration_count": 3, "approved": False}
print(f"Scenario 3 (rejected, iter=3): should_continue → '{should_continue(state_max)}'")
assert should_continue(state_max) == "done"  # Hard stop!

print("\n✅ All routing scenarios verified")

## Part 8: RAG Retrieval Quality Analysis

Let's explore how well the RAG retrieval is working by examining similarity scores.

In [ ]:
# Test RAG with several different queries and show similarity scores
test_queries = [
    "LangGraph features and capabilities",
    "production deployment best practices",
    "multi-agent system design patterns",
    "pizza recipe",  # out-of-domain test
]

print("RAG Retrieval Quality Test\n")
print("-" * 60)

for query in test_queries:
    docs_with_scores = rag.retrieve_with_scores(query)
    print(f"\nQuery: '{query}'")
    for j, (doc, score) in enumerate(docs_with_scores[:2], 1):
        # FAISS returns L2 distance — lower = more similar
        print(f"  Chunk {j} | Score: {score:.4f} | {doc.page_content[:80]}...")

## Part 9: Tool Testing

Test each tool independently to understand what agents have access to.

In [ ]:
from tools import search_web, rag_retrieve, calculate

# Test calculator (always works, no API key needed)
print("=== Calculator Tool ===")
expressions = ["100 * 1.35", "(2**10)", "3.14159 * 5**2"]
for expr in expressions:
    result = calculate.invoke(expr)
    print(f"  {expr} = {result}")

# Test security: reject unsafe expressions
print("\n  Security test (should reject):")
unsafe = calculate.invoke("__import__('os').system('echo hacked')")
print(f"  unsafe expression → {unsafe}")

In [ ]:
# Test web search
print("=== Web Search Tool ===")
web_result = search_web.invoke("LangGraph multi-agent AI framework")
print(web_result[:400])

## Part 10: Architecture Insights & Interview Prep

### Key Architecture Decisions Made in This Demo

In [ ]:
architecture_decisions = {
    "Why LangGraph over plain Python loops?": (
        "LangGraph provides checkpointing (resume on failure), streaming (real-time UI updates), "
        "parallel node execution, and built-in LangSmith tracing. "
        "Plain Python loops have none of these."
    ),
    "Why TypedDict for state?": (
        "TypedDict provides compile-time type checking without runtime overhead. "
        "It's serializable to JSON (enables checkpointing) and introspectable. "
        "Pydantic would add overhead; a plain dict would lose type safety."
    ),
    "Why FAISS instead of Pinecone for this demo?": (
        "FAISS is in-memory, requires no external service, and has zero setup. "
        "Perfect for demos and development. Swap to Pinecone in production with 1 line change."
    ),
    "Why temperature=0 for Planner/Analyst?": (
        "These nodes need deterministic, structured output (JSON). "
        "Temperature=0 minimizes randomness. "
        "The Writer gets 0.3 for more natural-sounding prose."
    ),
    "Why max_iterations=3 in the loop guard?": (
        "Without a hard stop, an LLM judge and LLM researcher can loop indefinitely "
        "(both are probabilistic). 3 iterations balances quality vs. cost/latency."
    ),
    "Why inject RAG pipeline via set_rag_pipeline()?": (
        "Dependency injection keeps the tools module decoupled from initialization order. "
        "It also makes testing easier — swap the real pipeline for a mock in unit tests."
    ),
}

for question, answer in architecture_decisions.items():
    print(f"Q: {question}")
    print(f"A: {answer}")
    print()

In [ ]:
# Interview prep: common questions about multi-agent systems
interview_questions = [
    "Design a multi-agent incident response system for a production outage.",
    "How would you evaluate whether your multi-agent pipeline is producing high-quality outputs?",
    "What are the tradeoffs between sequential pipelines and graph-based workflows?",
    "How would you reduce cost in a multi-agent system that makes 20+ LLM calls per query?",
    "How does LangGraph's checkpointing enable human-in-the-loop workflows?",
]

print("Top Interview Questions — Multi-Agent Systems\n")
print("=" * 50)
for i, q in enumerate(interview_questions, 1):
    print(f"{i}. {q}")

## Part 11: Try Your Own Query

Now try the full pipeline with your own research question!

In [ ]:
# ✏️ Change this to any research question you want!
MY_QUERY = "How does LangGraph compare to CrewAI for building multi-agent AI systems?"

# Run the pipeline
my_state = {
    "query": MY_QUERY,
    "plan": [],
    "research_results": [],
    "additional_queries": [],
    "analysis": "",
    "final_report": "",
    "iteration_count": 0,
    "approved": False,
}

print(f"Running pipeline for: '{MY_QUERY}'\n")

my_report = ""
for event in app.stream(my_state):
    node = list(event.keys())[0]
    output = event[node]
    print(f"✅ [{node.upper()}] done")
    if "final_report" in output and output["final_report"]:
        my_report = output["final_report"]

print("\n" + "=" * 60)
display(Markdown(my_report))

## Summary

### What We Built
- ✅ **GraphState** — shared typed state for all agents
- ✅ **RAGPipeline** — FAISS vector store with semantic search
- ✅ **4 Agent Nodes** — Planner, Researcher, Analyst, Writer
- ✅ **LangGraph Workflow** — conditional edges + self-correcting loop
- ✅ **Tools** — web search, RAG retrieval, calculator

### Key Takeaways
1. **Agents are just Python functions** — no magic, just structured LLM calls
2. **Shared state = shared whiteboard** — all agents read and write one dict
3. **Conditional edges = routing logic** — `should_continue()` decides the next step
4. **The Critic pattern** — an Analyst/Validator agent is what makes the pipeline reliable
5. **Loop guards are essential** — always add `max_iterations` to prevent infinite loops

### Next Steps
- Add a **Human-in-the-Loop** node using `interrupt()` for supervisor approval
- Connect **LangSmith** tracing to observe all LLM calls and token usage
- Run Researcher subtasks **in parallel** to reduce latency
- Deploy as a **FastAPI** endpoint with async streaming
- Add **evaluation** with a golden dataset and LLM-as-judge scoring